In [8]:
import pandas as pd
import numpy as np
import os

def topsis(input_file, weights, impacts, output_file):

    if not os.path.isfile(input_file):
        print("Error: File not found")
        return None

    data = pd.read_csv(input_file)

    # Remove old TOPSIS columns if present
    data = data.drop(columns=[col for col in data.columns
                               if col.lower() in ["d+", "d-", "topsis_score", "rank"]],
                     errors='ignore')

    if data.shape[1] < 3:
        print("Error: File must have at least 3 columns")
        return None

    criteria = data.iloc[:, 1:]

    for col in criteria.columns:
        if not pd.api.types.is_numeric_dtype(criteria[col]):
            print(f"Error: Column {col} must be numeric")
            return None

    weights = list(map(float, weights.split(",")))
    impacts = impacts.split(",")

    if len(weights) != len(impacts) or len(weights) != criteria.shape[1]:
        print("Error: Weights, impacts and criteria count must match")
        return None

    for impact in impacts:
        if impact not in ['+', '-']:
            print("Error: Impacts must be + or -")
            return None

    # Normalize
    norm = criteria / np.sqrt((criteria ** 2).sum())

    # Apply weights
    weighted = norm * weights

    # Ideal best & worst
    ideal_best = []
    ideal_worst = []

    for i in range(len(impacts)):
        if impacts[i] == '+':
            ideal_best.append(weighted.iloc[:, i].max())
            ideal_worst.append(weighted.iloc[:, i].min())
        else:
            ideal_best.append(weighted.iloc[:, i].min())
            ideal_worst.append(weighted.iloc[:, i].max())

    ideal_best = np.array(ideal_best)
    ideal_worst = np.array(ideal_worst)

    # Distances
    s_plus = np.sqrt(((weighted - ideal_best) ** 2).sum(axis=1))
    s_minus = np.sqrt(((weighted - ideal_worst) ** 2).sum(axis=1))

    # TOPSIS score
    score = s_minus / (s_plus + s_minus)

    data["TOPSIS_Score"] = score
    data["Rank"] = data["TOPSIS_Score"].rank(ascending=False).astype(int)

    data.to_csv(output_file, index=False)
    print("TOPSIS result saved to", output_file)

    return data


In [9]:
input_file = "TOPSIS_Fund_Ranking.csv"
weights = "1,1,1,1,1"
impacts = "+,+,+,+,+"
output_file = "TOPSIS_Result.csv"

result = topsis(input_file, weights, impacts, output_file)
result.head()


TOPSIS result saved to TOPSIS_Result.csv


,Fund,P1,P2,P3,P4,P5,TOPSIS_Score,Rank
0,M6,0.073559,0.074701,0.081771,0.078911,0.079581,0.738148,1
1,M5,0.078574,0.085373,0.045288,0.095306,0.090256,0.641886,2
2,M1,0.070215,0.068880,0.084287,0.064508,0.067198,0.563692,3
3,M2,0.076067,0.080522,0.088061,0.048572,0.053962,0.513032,4
4,M4,0.065200,0.059179,0.080513,0.064967,0.066985,0.491956,5


In [10]:
from google.colab import files
files.download("TOPSIS_Result.csv")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>